In [1]:
import torch

assert torch.cuda.is_available(), "No GPU. Runtime → Change runtime type → T4 GPU."

name = torch.cuda.get_device_name(0)
total = torch.cuda.get_device_properties(0).total_memory / 1024**3
free, _ = torch.cuda.mem_get_info()
free_gb = free / 1024**3

print(f"GPU: {name}")
print(f"Total VRAM: {total:.2f} GB")
print(f"Free VRAM before load: {free_gb:.2f} GB")

GPU: Tesla T4
Total VRAM: 14.56 GB
Free VRAM before load: 14.46 GB


In [3]:
!pip install -q -U transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.4 MB/s eta 0:00:00


In [2]:
import transformers, accelerate, bitsandbytes, torch
print(f"torch:        {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"accelerate:   {accelerate.__version__}")
print(f"bitsandbytes: {bitsandbytes.__version__}")

torch:        2.11.0+cu130
transformers: 5.18.0
accelerate:   1.15.0
bitsandbytes: 0.50.2


In [3]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))

In [5]:
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "NCAIR1/N-ATLaS"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Loading model at 4-bit...")
t0 = time.perf_counter()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
load_seconds = time.perf_counter() - t0
print(f"Model loaded in {load_seconds:.1f}s")

free, _ = torch.cuda.mem_get_info()
print(f"Free VRAM after load: {free / 1024**3:.2f} GB")
print(f"Model device: {next(model.parameters()).device}")

Loading tokenizer...


config.json:   0%|          | 0.00/875 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Model loaded in 553.5s
Free VRAM after load: 9.00 GB
Model device: cuda:0


In [6]:
import datetime

def format_prompt(messages):
    return tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
        date_string=datetime.datetime.now().strftime("%d %b %Y"),
    )

messages = [
    {
        "role": "system",
        "content": (
            "You are a patient secondary-school teacher in Nigeria. "
            "Explain clearly and simply."
        ),
    },
    {
        "role": "user",
        "content": "What is photosynthesis?",
    },
]

prompt = format_prompt(messages)
print("--- formatted prompt ---")
print(prompt[:500])
print("--- end ---")

inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to("cuda")

t0 = time.perf_counter()
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        use_cache=True,
        repetition_penalty=1.12,
        temperature=0.1,
        do_sample=True,
    )
elapsed = time.perf_counter() - t0

new_tokens = outputs.shape[-1] - inputs["input_ids"].shape[-1]
full_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
answer = full_text.split("assistant")[-1].strip()

print(f"elapsed_s: {elapsed:.2f}")
print(f"new_tokens: {new_tokens}")
print(f"tokens_per_sec: {new_tokens / elapsed:.2f}")
print("--- answer ---")
print(answer)

--- formatted prompt ---
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 05 Oct 2026

You are a patient secondary-school teacher in Nigeria. Explain clearly and simply.<|eot_id|><|start_header_id|>user<|end_header_id|>

What is photosynthesis?<|eot_id|><|start_header_id|>assistant<|end_header_id|>


--- end ---


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


elapsed_s: 16.26
new_tokens: 50
tokens_per_sec: 3.08
--- answer ---
Photosynthesis is how plants, algae, and some bacteria make their own food from sunlight. They use energy from the sun to turn water and carbon dioxide into glucose (a type of sugar) and oxygen. The oxygen they produce helps us breathe!


In [9]:
import json, datetime

result = {
    "timestamp_utc": datetime.datetime.utcnow().isoformat() + "Z",
    "model_id": MODEL_ID,
    "quantization": "4-bit nf4, double_quant=True, compute_dtype=float16",
    "gpu": torch.cuda.get_device_name(0),
    "vram_free_after_load_gb": round(torch.cuda.mem_get_info()[0] / 1024**3, 2),
    "transformers_version": transformers.__version__,
    "bitsandbytes_version": bitsandbytes.__version__,
    "torch_version": torch.__version__,
    "prompt": messages,
    "max_new_tokens": 300,
    "elapsed_seconds": round(elapsed, 2),
    "new_tokens": int(new_tokens),
    "tokens_per_sec": round(new_tokens / elapsed, 2),
    "answer": answer,
    "repo_commit": "096ee8618f70a70d8f4ea9d6ba39f3ab7eb9e983",
}

print(json.dumps(result, indent=2))

with open("natlas_llm_poc_result.json", "w") as f:
    json.dump(result, f, indent=2)

{
  "timestamp_utc": "2026-10-05T16:38:41.983793Z",
  "model_id": "NCAIR1/N-ATLaS",
  "quantization": "4-bit nf4, double_quant=True, compute_dtype=float16",
  "gpu": "Tesla T4",
  "vram_free_after_load_gb": 8.97,
  "transformers_version": "5.18.0",
  "bitsandbytes_version": "0.50.2",
  "torch_version": "2.11.0+cu130",
  "prompt": [
    {
      "role": "system",
      "content": "You are a patient secondary-school teacher in Nigeria. Explain clearly and simply."
    },
    {
      "role": "user",
      "content": "What is photosynthesis?"
    }
  ],
  "max_new_tokens": 300,
  "elapsed_seconds": 16.26,
  "new_tokens": 50,
  "tokens_per_sec": 3.08,
  "answer": "Photosynthesis is how plants, algae, and some bacteria make their own food from sunlight. They use energy from the sun to turn water and carbon dioxide into glucose (a type of sugar) and oxygen. The oxygen they produce helps us breathe!",
  "repo_commit": "096ee8618f70a70d8f4ea9d6ba39f3ab7eb9e983"
}


/tmp/ipykernel_13831/1148306078.py:4: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp_utc": datetime.datetime.utcnow().isoformat() + "Z",


In [10]:
from google.colab import files
files.download("natlas_llm_poc_result.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>